<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/research%2Fpost-e50-axial-level-pairing/68_postE50_lumbar_morphometric_measurements_FAST_RECOVERY_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 68 · Post-E50 — Lumbar Morphometric Measurements — **Recovery/Fast v2**

Versión autocontenida para continuar la etapa 68 sin depender del estado previo del runtime.

- Outer TRAIN únicamente.
- No entrenamiento.
- `internal_test` y test oficial permanecen bloqueados.
- Reutiliza checkpoints parciales existentes en Google Drive.
- 68-B y 68-C son resumibles.
- 68-C usa batching GPU y preflight liviano.
- Las clases axiales permanecen `raw_*`.
- 67C sigue **NOT VALIDATED** y requiere revisión humana.

**Uso:** abrir con NVIDIA L4 y ejecutar las celdas en orden. Si 68-B ya tiene manifest completo, no vuelve a inferir los 1382 estudios: sólo congela el CSV final.


In [1]:

# ============================================================
# 68-A — BOOTSTRAP AUTOCONTENIDO
# ============================================================

import hashlib, json, math, os, random, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
from typing import Mapping

import numpy as np
import pandas as pd

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

PFI_DRIVE_ROOT = Path(
    os.environ.get(
        "PFI_DRIVE_ROOT",
        "/content/drive/MyDrive/PFI_MVP" if IN_COLAB else "./PFI_MVP"
    )
)

RSNA_ROOT = PFI_DRIVE_ROOT / "data" / "RSNA_LUMBAR_DISC"
TRAIN_IMAGES_ROOT = RSNA_ROOT / "train_images"
RESULTS_DIR = PFI_DRIVE_ROOT / "results" / "post_e50" / "68"
METRICS_DIR = PFI_DRIVE_ROOT / "metrics" / "post_e50" / "68"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

SAGITTAL_CHECKPOINT_PATH = (
    PFI_DRIVE_ROOT / "models" / "final" /
    "sagittal_spider_multiclass_final_best_cf11dcc0.pt"
)

AXIAL_CHECKPOINT_PATH = (
    PFI_DRIVE_ROOT / "models" / "final" /
    "axial_t2_alkafri_final_v2_candidate.pt"
)

EXPECTED_SAGITTAL_SHA256 = (
    "cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944"
)

EXPECTED_AXIAL_SHA256 = (
    "a48cbddd858b5615010fd809412f3d17dae6871fbe12a38f4720e6f6bc70f739"
)

EXPECTED_CSV_SHA256 = {
    "train.csv": "f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528",
    "train_label_coordinates.csv": "416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635",
    "train_series_descriptions.csv": "bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33",
}

SAGITTAL_CLASS_NAMES = {
    0: "background",
    1: "vertebra_group",
    2: "canal",
    3: "disc_group",
}

AXIAL_CLASS_NAMES = {
    0: "background_250",
    1: "raw_0",
    2: "raw_50",
    3: "raw_100",
    4: "raw_150",
    5: "raw_200",
}

RSNA_INTERNAL_TEST_LOCKED = True
OFFICIAL_TEST_ACCESSED = False
TRAINING_PERFORMED = False

assert RSNA_INTERNAL_TEST_LOCKED
assert not OFFICIAL_TEST_ACCESSED
assert not TRAINING_PERFORMED

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def opaque_id(x):
    return hashlib.sha256(str(x).encode()).hexdigest()[:12]

def build_split(ids, seed=2026):
    ordered = sorted(str(x) for x in ids)
    rng = np.random.default_rng(seed)
    sh = list(ordered)
    rng.shuffle(sh)
    n = len(sh)
    a = int(n * .70)
    b = int(n * .15)
    return {
        "train": sorted(sh[:a]),
        "validation": sorted(sh[a:a+b]),
        "internal_test": sorted(sh[a+b:]),
    }

for fn, expected in EXPECTED_CSV_SHA256.items():
    p = RSNA_ROOT / fn
    assert p.is_file(), p
    got = sha256_file(p)
    print(fn, got, "MATCH" if got == expected else "MISMATCH")
    assert got == expected

assert SAGITTAL_CHECKPOINT_PATH.is_file(), SAGITTAL_CHECKPOINT_PATH
assert AXIAL_CHECKPOINT_PATH.is_file(), AXIAL_CHECKPOINT_PATH

sag_sha = sha256_file(SAGITTAL_CHECKPOINT_PATH)
ax_sha = sha256_file(AXIAL_CHECKPOINT_PATH)

assert sag_sha == EXPECTED_SAGITTAL_SHA256, sag_sha
assert ax_sha == EXPECTED_AXIAL_SHA256, ax_sha

series_df = pd.read_csv(
    RSNA_ROOT / "train_series_descriptions.csv",
    dtype={"study_id": "string", "series_id": "string"},
)

outer_split = build_split(series_df.study_id.dropna().unique())
assert {k: len(v) for k, v in outer_split.items()} == {
    "train": 1382,
    "validation": 296,
    "internal_test": 297,
}

RUN_STUDY_IDS = outer_split["train"]

print("Outer TRAIN:", len(RUN_STUDY_IDS))
print("Sag SHA:", sag_sha)
print("Ax SHA:", ax_sha)
print("68-A BOOTSTRAP: PASS")


Mounted at /content/drive
train.csv f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528 MATCH
train_label_coordinates.csv 416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635 MATCH
train_series_descriptions.csv bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33 MATCH
Outer TRAIN: 1382
Sag SHA: cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944
Ax SHA: a48cbddd858b5615010fd809412f3d17dae6871fbe12a38f4720e6f6bc70f739
68-A BOOTSTRAP: PASS


In [2]:

# ============================================================
# RUNTIME — LIBRERÍAS, MODELOS Y HELPERS
# ============================================================

def _importable(name):
    try:
        m = __import__(name)
        return getattr(m, "__version__", "unknown")
    except ImportError:
        return None

pkgs = []
if _importable("pydicom") is None:
    pkgs.append("pydicom")
if _importable("scipy") is None:
    pkgs.append("scipy")
if _importable("PIL") is None:
    pkgs.append("Pillow")

if pkgs:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *pkgs]
    )

import torch
import torch.nn as nn
import pydicom

from PIL import Image
from scipy import ndimage
from pandas.errors import EmptyDataError

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

class SDC(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1),
            nn.BatchNorm2d(o),
            nn.ReLU(True),
            nn.Conv2d(o, o, 3, padding=1),
            nn.BatchNorm2d(o),
            nn.ReLU(True),
        )
    def forward(self, x):
        return self.block(x)

class SagUNet(nn.Module):
    def __init__(self, num_classes=4, base_channels=16):
        super().__init__()
        b = base_channels
        self.enc1 = SDC(1, b)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = SDC(b, b*2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = SDC(b*2, b*4)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = SDC(b*4, b*8)
        self.up3 = nn.ConvTranspose2d(b*8, b*4, 2, 2)
        self.dec3 = SDC(b*8, b*4)
        self.up2 = nn.ConvTranspose2d(b*4, b*2, 2, 2)
        self.dec2 = SDC(b*4, b*2)
        self.up1 = nn.ConvTranspose2d(b*2, b, 2, 2)
        self.dec1 = SDC(b*2, b)
        self.out_conv = nn.Conv2d(b, num_classes, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        z = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(z), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.out_conv(d1)

class ADC(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1),
            nn.BatchNorm2d(o),
            nn.ReLU(True),
            nn.Conv2d(o, o, 3, padding=1),
            nn.BatchNorm2d(o),
            nn.ReLU(True),
        )
    def forward(self, x):
        return self.net(x)

class AxUNet(nn.Module):
    def __init__(self, num_classes=6, base_channels=16):
        super().__init__()
        b = base_channels
        self.e1 = ADC(1, b)
        self.e2 = ADC(b, b*2)
        self.e3 = ADC(b*2, b*4)
        self.pool = nn.MaxPool2d(2)
        self.mid = ADC(b*4, b*8)
        self.u3 = nn.ConvTranspose2d(b*8, b*4, 2, 2)
        self.d3 = ADC(b*8, b*4)
        self.u2 = nn.ConvTranspose2d(b*4, b*2, 2, 2)
        self.d2 = ADC(b*4, b*2)
        self.u1 = nn.ConvTranspose2d(b*2, b, 2, 2)
        self.d1 = ADC(b*2, b)
        self.out = nn.Conv2d(b, num_classes, 1)
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        z = self.mid(self.pool(e3))
        x = self.d3(torch.cat([self.u3(z), e3], 1))
        x = self.d2(torch.cat([self.u2(x), e2], 1))
        x = self.d1(torch.cat([self.u1(x), e1], 1))
        return self.out(x)

def _state(ck):
    if not isinstance(ck, Mapping):
        raise ValueError("checkpoint")
    for k in ("model_state_dict", "state_dict", "model"):
        if isinstance(ck.get(k), Mapping):
            sd = ck[k]
            break
    else:
        sd = ck
    out = {}
    for k, v in sd.items():
        kk = str(k)
        for prefix in ("module.", "model."):
            if kk.startswith(prefix):
                kk = kk[len(prefix):]
        out[kk] = v
    return out

def load_model(path, kind):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    sd = _state(ck)
    if kind == "sag":
        b = int(ck.get("base_channels", sd["enc1.block.0.weight"].shape[0]))
        n = int(ck.get("num_classes", sd["out_conv.weight"].shape[0]))
        m = SagUNet(n, b)
    else:
        b = int(ck.get("base_channels", sd["e1.net.0.weight"].shape[0]))
        n = int(ck.get("num_classes", sd["out.weight"].shape[0]))
        m = AxUNet(n, b)
    m.load_state_dict(sd, strict=True)
    m.to(DEVICE).eval()
    target = tuple(int(x) for x in ck.get("target_size", (256, 256)))
    return m, target

sag_model, SAG_TARGET = load_model(SAGITTAL_CHECKPOINT_PATH, "sag")
ax_model, AX_TARGET = load_model(AXIAL_CHECKPOINT_PATH, "ax")

def robust_norm(a):
    x = np.asarray(a, dtype=np.float32)
    finite = np.isfinite(x)
    if not finite.any():
        return np.zeros_like(x)
    lo, hi = np.percentile(x[finite], [1, 99])
    if hi <= lo:
        return np.zeros_like(x)
    return np.clip((x-lo)/(hi-lo+1e-8), 0, 1).astype(np.float32)

def comps(binary, min_pixels=20):
    lab, n = ndimage.label(binary.astype(np.uint8))
    out = []
    for i in range(1, n+1):
        m = lab == i
        if int(m.sum()) >= min_pixels:
            out.append(m)
    return sorted(out, key=lambda m: int(m.sum()), reverse=True)

def morph(mask, spacing):
    if spacing is None or not mask.any():
        return None
    rs, cs = map(float, spacing)
    rr, cc = np.where(mask)
    pts = np.column_stack([rr*rs, cc*cs]).astype(float)
    pts -= pts.mean(0)
    if len(pts) >= 3:
        eig = np.linalg.eigh(np.cov(pts, rowvar=False))[1]
        pr = pts @ eig
        lengths = np.ptp(pr, axis=0)
    else:
        lengths = np.array([0.0, 0.0])
    major, minor = sorted(map(float, lengths), reverse=True)
    return {
        "area_mm2": float(mask.sum()) * rs * cs,
        "bbox_height_mm": float(rr.max()-rr.min()+1) * rs,
        "bbox_width_mm": float(cc.max()-cc.min()+1) * cs,
        "major_axis_mm": major,
        "minor_axis_mm": minor,
        "centroid_row_px": float(rr.mean()),
        "centroid_col_px": float(cc.mean()),
        "pixel_count": int(mask.sum()),
    }

def safe_csv(path):
    if not Path(path).is_file():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except EmptyDataError:
        return pd.DataFrame()

def numeric_dicom_paths(study_id, series_id):
    root = TRAIN_IMAGES_ROOT / str(study_id) / str(series_id)
    paths = list(root.glob("*.dcm"))
    def key(p):
        try:
            return (0, int(p.stem))
        except ValueError:
            return (1, p.name)
    return sorted(paths, key=key)

def prepare_model_input(arr, target):
    x = robust_norm(arr)
    img = Image.fromarray((x*255).astype(np.uint8))
    img = img.resize((target[1], target[0]), Image.Resampling.BILINEAR)
    return np.asarray(img, dtype=np.float32) / 255.0

def batch_predict(model, arrays, target):
    inputs = np.stack(
        [prepare_model_input(arr, target) for arr in arrays],
        axis=0,
    )
    tensor = torch.from_numpy(inputs[:, None]).to(DEVICE, non_blocking=True)
    with torch.inference_mode():
        pred = model(tensor).argmax(1).cpu().numpy().astype(np.uint8)
    return pred

def resize_mask_native(mask, native_shape):
    img = Image.fromarray(mask.astype(np.uint8))
    img = img.resize(
        (native_shape[1], native_shape[0]),
        Image.Resampling.NEAREST
    )
    return np.asarray(img, dtype=np.uint8)

print("Models loaded:", SAG_TARGET, AX_TARGET)
print("Runtime helpers: PASS")


DEVICE: cuda
GPU: NVIDIA L4
Models loaded: (256, 256) (256, 256)
Runtime helpers: PASS


In [3]:

# ============================================================
# 68-B FAST — RECOVER / FINALIZE / RESUME
# ============================================================

assert torch.cuda.is_available(), "68-B requiere GPU/L4."

BATCH_SIZE_B = 32
CHECKPOINT_EVERY_B = 100

B_PARTIAL_PATH = RESULTS_DIR / "68_B_sagittal_measurements.partial.csv"
B_FAIL_PARTIAL_PATH = RESULTS_DIR / "68_B_sagittal_failures.partial.csv"
B_DONE_PATH = RESULTS_DIR / "68_B_done_studies.json"

SAG_PATH = RESULTS_DIR / "68_B_sagittal_measurements.csv"
SAG_FAIL_PATH = RESULTS_DIR / "68_B_sagittal_failures.csv"

sag_partial = safe_csv(B_PARTIAL_PATH)
sag_fail_partial = safe_csv(B_FAIL_PARTIAL_PATH)

valid_opaque = {opaque_id(sid) for sid in RUN_STUDY_IDS}

# El manifest es autoritativo porque la versión FAST lo escribe sólo
# después de marcar estudios procesados. Esto permite estudios válidos
# con 0 componentes y 0 filas.
if B_DONE_PATH.is_file():
    try:
        done_b = set(json.loads(B_DONE_PATH.read_text(encoding="utf-8")))
    except Exception:
        done_b = set()
else:
    done_b = set()

done_b &= valid_opaque

# Fallback si no hay manifest.
if not done_b:
    if not sag_partial.empty:
        done_b.update(
            sag_partial["study_id_opaque"].dropna().astype(str).tolist()
        )
    if not sag_fail_partial.empty:
        done_b.update(
            sag_fail_partial["study_id_opaque"].dropna().astype(str).tolist()
        )

sdf = series_df[
    series_df.study_id.astype(str).isin(set(map(str, RUN_STUDY_IDS)))
    &
    (series_df.series_description.astype(str).str.strip() == "Sagittal T2/STIR")
].copy()

series_by_study_b = {}
for _, row in sdf.iterrows():
    sid = str(row.study_id)
    if sid not in series_by_study_b:
        series_by_study_b[sid] = str(row.series_id)

remaining_b = [
    sid for sid in RUN_STUDY_IDS
    if opaque_id(sid) not in done_b
]

print("="*72)
print("68-B FAST RECOVERY")
print("="*72)
print("Total:", len(RUN_STUDY_IDS))
print("Recovered completed:", len(done_b))
print("Remaining:", len(remaining_b))

def save_b():
    if not sag_partial.empty:
        sag_partial.to_csv(B_PARTIAL_PATH, index=False)
    if not sag_fail_partial.empty:
        sag_fail_partial.to_csv(B_FAIL_PARTIAL_PATH, index=False)
    elif B_FAIL_PARTIAL_PATH.exists():
        B_FAIL_PARTIAL_PATH.unlink()

    B_DONE_PATH.write_text(
        json.dumps(sorted(done_b), indent=2),
        encoding="utf-8"
    )

if remaining_b:
    t0 = time.time()
    session_n = 0

    for start in range(0, len(remaining_b), BATCH_SIZE_B):
        batch_ids = remaining_b[start:start+BATCH_SIZE_B]
        loaded = []

        for sid in batch_ids:
            oid = opaque_id(sid)
            try:
                ser = series_by_study_b.get(str(sid))
                if ser is None:
                    raise RuntimeError("NO_SAGITTAL_T2")

                paths = numeric_dicom_paths(sid, ser)
                if not paths:
                    raise RuntimeError("NO_DICOM")

                p = paths[len(paths)//2]
                ds = pydicom.dcmread(str(p))
                arr = ds.pixel_array.astype(np.float32)

                if not hasattr(ds, "PixelSpacing"):
                    raise RuntimeError("MISSING_PIXEL_SPACING")

                spacing = tuple(float(v) for v in ds.PixelSpacing)
                loaded.append({
                    "sid": sid, "oid": oid, "ser": ser,
                    "ds": ds, "arr": arr, "spacing": spacing,
                })

            except Exception as exc:
                sag_fail_partial = pd.concat([
                    sag_fail_partial,
                    pd.DataFrame([{
                        "study_id_opaque": oid,
                        "reason": f"{type(exc).__name__}:{exc}",
                    }])
                ], ignore_index=True)
                done_b.add(oid)
                session_n += 1

        if loaded:
            preds = batch_predict(
                sag_model,
                [x["arr"] for x in loaded],
                SAG_TARGET,
            )

            new_rows = []

            for item, pred_small in zip(loaded, preds):
                oid = item["oid"]
                try:
                    pred = resize_mask_native(
                        pred_small,
                        item["arr"].shape,
                    )

                    for cls, name, minpix in [
                        (3, "disc_group_component", 20),
                        (2, "canal", 20),
                        (1, "vertebra_group_component", 50),
                    ]:
                        cc = comps(pred == cls, minpix)

                        if cls == 2:
                            cc = cc[:1]
                        else:
                            cc = sorted(
                                cc,
                                key=lambda m: float(np.where(m)[0].mean())
                            )

                        for rank, c in enumerate(cc, 1):
                            mm = morph(c, item["spacing"])
                            new_rows.append({
                                "study_id_opaque": oid,
                                "series_id_opaque": opaque_id(item["ser"]),
                                "instance_number": int(
                                    getattr(item["ds"], "InstanceNumber", 0)
                                ),
                                "structure": name,
                                "component_rank_superior_to_inferior_proxy":
                                    rank if cls != 2 else None,
                                **mm,
                            })

                except Exception as exc:
                    sag_fail_partial = pd.concat([
                        sag_fail_partial,
                        pd.DataFrame([{
                            "study_id_opaque": oid,
                            "reason": f"{type(exc).__name__}:{exc}",
                        }])
                    ], ignore_index=True)

                # IMPORTANTE: incluso 0 componentes cuenta como procesado.
                done_b.add(oid)
                session_n += 1

            if new_rows:
                sag_partial = pd.concat(
                    [sag_partial, pd.DataFrame(new_rows)],
                    ignore_index=True,
                )

        if session_n and session_n % CHECKPOINT_EVERY_B < BATCH_SIZE_B:
            save_b()
            elapsed = (time.time() - t0) / 60
            print(
                f"68-B checkpoint | {len(done_b)}/{len(RUN_STUDY_IDS)} "
                f"| elapsed={elapsed:.1f} min"
            )

    save_b()

assert len(done_b) == len(RUN_STUDY_IDS), (
    f"68-B incomplete: {len(done_b)}/{len(RUN_STUDY_IDS)}"
)

sag_df = (
    sag_partial
    .drop_duplicates()
    .sort_values(["study_id_opaque", "instance_number", "structure"])
    .reset_index(drop=True)
)

sag_fail_df = sag_fail_partial.drop_duplicates().reset_index(drop=True)

sag_df.to_csv(SAG_PATH, index=False)
sag_fail_df.to_csv(SAG_FAIL_PATH, index=False)

print("="*72)
print("68-B FAST COMPLETE")
print("="*72)
print("Studies completed:", len(done_b))
print("Studies with measurements:", sag_df.study_id_opaque.nunique())
print("Rows:", len(sag_df))
print("Failures:", len(sag_fail_df))
print("SHA256:", sha256_file(SAG_PATH))


68-B FAST RECOVERY
Total: 1382
Recovered completed: 1382
Remaining: 0
68-B FAST COMPLETE
Studies completed: 1382
Studies with measurements: 1380
Rows: 30085
Failures: 1
SHA256: 35f4426de9bf508cafa746e9307d4d93f515a6e444b90a491a264aa52269c97e


In [4]:

# ============================================================
# 68-C FAST — RESUMABLE AXIAL RAW-CLASS MEASUREMENTS
# ============================================================

assert torch.cuda.is_available(), "68-C requiere GPU/L4."

BATCH_SIZE_C = 64
CHECKPOINT_EVERY_C = 200

C_PARTIAL_PATH = RESULTS_DIR / "68_C_axial_rawclass_measurements.partial.csv"
C_FAIL_PARTIAL_PATH = RESULTS_DIR / "68_C_axial_failures.partial.csv"
C_DONE_PATH = RESULTS_DIR / "68_C_done_studies.json"

AX_PATH = RESULTS_DIR / "68_C_axial_rawclass_measurements.csv"
AX_FAIL_PATH = RESULTS_DIR / "68_C_axial_failures.csv"

ax_partial = safe_csv(C_PARTIAL_PATH)
ax_fail_partial = safe_csv(C_FAIL_PARTIAL_PATH)

valid_opaque = {opaque_id(sid) for sid in RUN_STUDY_IDS}

if C_DONE_PATH.is_file():
    try:
        done_c = set(json.loads(C_DONE_PATH.read_text(encoding="utf-8")))
    except Exception:
        done_c = set()
else:
    done_c = set()

done_c &= valid_opaque

if not done_c:
    if not ax_partial.empty:
        done_c.update(
            ax_partial["study_id_opaque"].dropna().astype(str).tolist()
        )
    if not ax_fail_partial.empty:
        done_c.update(
            ax_fail_partial["study_id_opaque"].dropna().astype(str).tolist()
        )

adf = series_df[
    series_df.study_id.astype(str).isin(set(map(str, RUN_STUDY_IDS)))
    &
    series_df.series_description.astype(str)
    .str.upper()
    .str.contains("AXIAL", na=False)
].copy()

series_by_study_c = {}
for _, row in adf.iterrows():
    series_by_study_c.setdefault(str(row.study_id), []).append(str(row.series_id))

remaining_c = [
    sid for sid in RUN_STUDY_IDS
    if opaque_id(sid) not in done_c
]

print("="*72)
print("68-C FAST RESUMABLE START")
print("="*72)
print("Total outer TRAIN:", len(RUN_STUDY_IDS))
print("Recovered completed:", len(done_c))
print("Remaining:", len(remaining_c))
print("GPU batch size:", BATCH_SIZE_C)

# ------------------------------------------------------------
# LIGHT PREFLIGHT: sólo si hay trabajo pendiente
# ------------------------------------------------------------

if remaining_c:
    preflight_pairs = []
    for sid in remaining_c:
        for ser in series_by_study_c.get(str(sid), []):
            preflight_pairs.append((sid, ser))
            if len(preflight_pairs) >= 3:
                break
        if len(preflight_pairs) >= 3:
            break

    checked = 0

    for sid, ser in preflight_pairs:
        paths = numeric_dicom_paths(sid, ser)
        assert paths, (opaque_id(sid), opaque_id(ser))

        sample_idx = sorted(set([0, len(paths)//2, len(paths)-1]))

        for idx in sample_idx:
            p = paths[idx]
            ds = pydicom.dcmread(
                str(p),
                stop_before_pixels=True,
                specific_tags=["InstanceNumber"],
            )
            instance_number = int(getattr(ds, "InstanceNumber", -1))
            filename_number = int(p.stem)
            assert filename_number == instance_number, (
                "HARD STOP filename != InstanceNumber",
                opaque_id(sid),
                opaque_id(ser),
                p.name,
                instance_number,
            )
            checked += 1

    print("Light preflight sampled headers:", checked)
    print("filename == InstanceNumber: PASS")

def save_c():
    if not ax_partial.empty:
        ax_partial.to_csv(C_PARTIAL_PATH, index=False)
    if not ax_fail_partial.empty:
        ax_fail_partial.to_csv(C_FAIL_PARTIAL_PATH, index=False)
    elif C_FAIL_PARTIAL_PATH.exists():
        C_FAIL_PARTIAL_PATH.unlink()

    C_DONE_PATH.write_text(
        json.dumps(sorted(done_c), indent=2),
        encoding="utf-8",
    )

# Estudios sin axial: se marcan failure + done.
work_items = []
pending = {}

for sid in remaining_c:
    oid = opaque_id(sid)
    series_list = series_by_study_c.get(str(sid), [])

    if not series_list:
        ax_fail_partial = pd.concat([
            ax_fail_partial,
            pd.DataFrame([{
                "study_id_opaque": oid,
                "reason": "NO_AXIAL_SERIES",
            }])
        ], ignore_index=True)
        done_c.add(oid)
        continue

    pending[oid] = len(series_list)

    for ser in series_list:
        work_items.append({
            "sid": sid,
            "oid": oid,
            "ser": ser,
        })

print("Axial series to process:", len(work_items))

t0 = time.time()
processed_series = 0
last_saved_done = len(done_c)

for start in range(0, len(work_items), BATCH_SIZE_C):
    batch_items = work_items[start:start+BATCH_SIZE_C]
    loaded = []

    for item in batch_items:
        sid, oid, ser = item["sid"], item["oid"], item["ser"]

        try:
            paths = numeric_dicom_paths(sid, ser)
            if not paths:
                raise RuntimeError("NO_DICOM")

            p = paths[len(paths)//2]
            ds = pydicom.dcmread(str(p))
            arr = ds.pixel_array.astype(np.float32)

            if not hasattr(ds, "PixelSpacing"):
                raise RuntimeError("MISSING_PIXEL_SPACING")

            spacing = tuple(float(v) for v in ds.PixelSpacing)

            loaded.append({
                **item,
                "ds": ds,
                "arr": arr,
                "spacing": spacing,
            })

        except Exception as exc:
            ax_fail_partial = pd.concat([
                ax_fail_partial,
                pd.DataFrame([{
                    "study_id_opaque": oid,
                    "series_id_opaque": opaque_id(ser),
                    "reason": f"{type(exc).__name__}:{exc}",
                }])
            ], ignore_index=True)

            pending[oid] -= 1
            processed_series += 1

            if pending[oid] == 0:
                done_c.add(oid)

    if loaded:
        preds = batch_predict(
            ax_model,
            [x["arr"] for x in loaded],
            AX_TARGET,
        )

        new_rows = []

        for item, pred_small in zip(loaded, preds):
            oid, ser = item["oid"], item["ser"]

            try:
                pred = resize_mask_native(
                    pred_small,
                    item["arr"].shape,
                )

                for cls in range(1, 6):
                    name = AXIAL_CLASS_NAMES[cls]
                    union = pred == cls
                    cc = comps(union, 10)

                    if cc:
                        new_rows.append({
                            "study_id_opaque": oid,
                            "series_id_opaque": opaque_id(ser),
                            "instance_number": int(
                                getattr(item["ds"], "InstanceNumber", 0)
                            ),
                            "class_id": cls,
                            "class_name": name,
                            "measurement_scope": "largest_component",
                            "anatomical_name_confirmed": False,
                            "axial_level_pairing_validated": False,
                            **morph(cc[0], item["spacing"]),
                        })

                    if union.any():
                        new_rows.append({
                            "study_id_opaque": oid,
                            "series_id_opaque": opaque_id(ser),
                            "instance_number": int(
                                getattr(item["ds"], "InstanceNumber", 0)
                            ),
                            "class_id": cls,
                            "class_name": name,
                            "measurement_scope": "class_union",
                            "anatomical_name_confirmed": False,
                            "axial_level_pairing_validated": False,
                            **morph(union, item["spacing"]),
                        })

            except Exception as exc:
                ax_fail_partial = pd.concat([
                    ax_fail_partial,
                    pd.DataFrame([{
                        "study_id_opaque": oid,
                        "series_id_opaque": opaque_id(ser),
                        "reason": f"{type(exc).__name__}:{exc}",
                    }])
                ], ignore_index=True)

            pending[oid] -= 1
            processed_series += 1

            if pending[oid] == 0:
                done_c.add(oid)

        if new_rows:
            ax_partial = pd.concat(
                [ax_partial, pd.DataFrame(new_rows)],
                ignore_index=True,
            )

    if len(done_c) - last_saved_done >= CHECKPOINT_EVERY_C:
        save_c()
        elapsed = (time.time() - t0) / 60
        done_session = len(done_c) - (
            len(RUN_STUDY_IDS) - len(remaining_c)
        )
        rate = done_session / max(elapsed, 1e-9)
        eta = (len(RUN_STUDY_IDS) - len(done_c)) / max(rate, 1e-9)

        print(
            f"68-C checkpoint | studies={len(done_c)}/{len(RUN_STUDY_IDS)} "
            f"| series={processed_series}/{len(work_items)} "
            f"| {rate:.1f} studies/min | ETA={eta:.1f} min"
        )

        last_saved_done = len(done_c)

save_c()

assert len(done_c) == len(RUN_STUDY_IDS), (
    f"68-C incomplete: {len(done_c)}/{len(RUN_STUDY_IDS)}"
)

ax_df = (
    ax_partial
    .drop_duplicates()
    .sort_values([
        "study_id_opaque",
        "series_id_opaque",
        "instance_number",
        "class_id",
        "measurement_scope",
    ])
    .reset_index(drop=True)
)

ax_fail_df = ax_fail_partial.drop_duplicates().reset_index(drop=True)

ax_df.to_csv(AX_PATH, index=False)
ax_fail_df.to_csv(AX_FAIL_PATH, index=False)

print("="*72)
print("68-C FAST COMPLETE")
print("="*72)
print("Studies completed:", len(done_c))
print("Studies with measurements:", ax_df.study_id_opaque.nunique())
print("Rows:", len(ax_df))
print("Failures:", len(ax_fail_df))
print("SHA256:", sha256_file(AX_PATH))
print("Anatomical names confirmed: False")
print("67C pairing formally validated: False")


68-C FAST RESUMABLE START
Total outer TRAIN: 1382
Recovered completed: 0
Remaining: 1382
GPU batch size: 32
Light preflight sampled headers: 9
filename == InstanceNumber: PASS
Axial series to process: 1632
68-C checkpoint | studies=106/1382 | series=128/1632 | 28.3 studies/min | ETA=45.2 min
68-C checkpoint | studies=219/1382 | series=256/1632 | 29.3 studies/min | ETA=39.7 min
68-C checkpoint | studies=322/1382 | series=384/1632 | 28.4 studies/min | ETA=37.4 min
68-C checkpoint | studies=430/1382 | series=512/1632 | 27.9 studies/min | ETA=34.1 min
68-C checkpoint | studies=541/1382 | series=640/1632 | 28.1 studies/min | ETA=29.9 min
68-C checkpoint | studies=648/1382 | series=768/1632 | 27.8 studies/min | ETA=26.4 min
68-C checkpoint | studies=756/1382 | series=896/1632 | 27.7 studies/min | ETA=22.6 min
68-C checkpoint | studies=864/1382 | series=1024/1632 | 27.2 studies/min | ETA=19.0 min
68-C checkpoint | studies=976/1382 | series=1152/1632 | 27.2 studies/min | ETA=14.9 min
68-C chec

In [5]:

# ============================================================
# 68-D — TECHNICAL QC AUTOCONTENIDO
# ============================================================

SAG_PATH = RESULTS_DIR / "68_B_sagittal_measurements.csv"
SAG_FAIL_PATH = RESULTS_DIR / "68_B_sagittal_failures.csv"
AX_PATH = RESULTS_DIR / "68_C_axial_rawclass_measurements.csv"
AX_FAIL_PATH = RESULTS_DIR / "68_C_axial_failures.csv"

assert SAG_PATH.is_file(), SAG_PATH
assert AX_PATH.is_file(), AX_PATH

sag_df = safe_csv(SAG_PATH)
ax_df = safe_csv(AX_PATH)
sag_fail_df = safe_csv(SAG_FAIL_PATH)
ax_fail_df = safe_csv(AX_FAIL_PATH)

def audit(df, name):
    if df.empty:
        return {"name": name, "rows": 0, "status": "NO_ROWS"}

    cols = [
        "area_mm2",
        "bbox_height_mm",
        "bbox_width_mm",
        "major_axis_mm",
        "minor_axis_mm",
    ]

    n = df[cols].apply(pd.to_numeric, errors="coerce")
    finite = np.isfinite(n.to_numpy()).all(1)
    pos = (
        (n.area_mm2 > 0)
        & (n.bbox_height_mm > 0)
        & (n.bbox_width_mm > 0)
    )
    axis = n.major_axis_mm + 1e-9 >= n.minor_axis_mm

    return {
        "name": name,
        "rows": int(len(df)),
        "finite_rate": float(finite.mean()),
        "positive_rate": float(pos.mean()),
        "major_ge_minor_rate": float(axis.mean()),
        "status": (
            "PASS"
            if finite.all() and pos.all() and axis.all()
            else "FAIL"
        ),
    }

sag_qc = audit(sag_df, "sagittal")
ax_qc = audit(ax_df, "axial_raw_classes")

for d in (sag_df, ax_df):
    if not d.empty:
        assert "study_id" not in d.columns
        assert "series_id" not in d.columns
        assert "study_id_opaque" in d.columns
        assert "series_id_opaque" in d.columns

# Independiente de que morph exista o no.
mask = np.ones((4, 4), dtype=bool)
assert abs(float(mask.sum()) * 1.0 * 1.0 - 16.0) < 1e-9
assert abs(float(mask.sum()) * 2.0 * 0.5 - 16.0) < 1e-9

qc = {
    "stage": "68-D",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "run_studies": len(RUN_STUDY_IDS),
    "scope": "OUTER_TRAIN_ONLY",
    "sagittal_measurement_studies": (
        int(sag_df.study_id_opaque.nunique()) if not sag_df.empty else 0
    ),
    "axial_measurement_studies": (
        int(ax_df.study_id_opaque.nunique()) if not ax_df.empty else 0
    ),
    "sagittal_failures": int(len(sag_fail_df)),
    "axial_failures": int(len(ax_fail_df)),
    "sagittal_qc": sag_qc,
    "axial_qc": ax_qc,
    "pixel_spacing_invariance": "PASS",
    "internal_test_accessed": False,
    "official_test_accessed": False,
    "training_performed": False,
}

QC_PATH = METRICS_DIR / "68_D_qc_summary.json"
QC_PATH.write_text(json.dumps(qc, indent=2))

print("="*72)
print("68-D TECHNICAL QC")
print("="*72)
print("Sag studies with measurements:", qc["sagittal_measurement_studies"])
print("Sag failures:", len(sag_fail_df))
print("Ax studies with measurements:", qc["axial_measurement_studies"])
print("Ax failures:", len(ax_fail_df))
print("Sagittal QC:", sag_qc)
print("Axial QC:", ax_qc)
print("PixelSpacing invariance: PASS")
print("SHA256:", sha256_file(QC_PATH))


68-D TECHNICAL QC
Sag studies with measurements: 1380
Sag failures: 1
Ax studies with measurements: 1382
Ax failures: 0
Sagittal QC: {'name': 'sagittal', 'rows': 30085, 'finite_rate': 1.0, 'positive_rate': 1.0, 'major_ge_minor_rate': 1.0, 'status': 'PASS'}
Axial QC: {'name': 'axial_raw_classes', 'rows': 15107, 'finite_rate': 1.0, 'positive_rate': 1.0, 'major_ge_minor_rate': 1.0, 'status': 'PASS'}
PixelSpacing invariance: PASS
SHA256: 006f6d4392c6847ed2d974ba5ae84b48ca878bf0e5c7478d591ac2889531a4f4


In [6]:

# ============================================================
# 68-E — FINAL CONTRACT
# ============================================================

# Restore checkpoint identities explicitly.
sag_sha = sha256_file(SAGITTAL_CHECKPOINT_PATH)
ax_sha = sha256_file(AXIAL_CHECKPOINT_PATH)

assert sag_sha == EXPECTED_SAGITTAL_SHA256
assert ax_sha == EXPECTED_AXIAL_SHA256

contract = {
    "contract_version": "68.2",
    "purpose": "NON_DIAGNOSTIC_REVIEWABLE_LUMBAR_MORPHOMETRY",
    "units": {
        "linear": "mm",
        "area": "mm2",
    },
    "sagittal": {
        "source_model": "sagittal_spider",
        "checkpoint_sha256": sag_sha,
        "semantic_classes": SAGITTAL_CLASS_NAMES,
        "measurements": [
            "area_mm2",
            "bbox_height_mm",
            "bbox_width_mm",
            "major_axis_mm",
            "minor_axis_mm",
        ],
        "absolute_level_source_runtime": "67B2",
        "absolute_level_status": "VALIDATED_WITH_ABSTENTION",
        "human_review_required": True,
    },
    "axial": {
        "source_model": "axial_t2_alkafri_final_v2_candidate",
        "checkpoint_sha256": ax_sha,
        "semantic_classes": AXIAL_CLASS_NAMES,
        "measurements": [
            "area_mm2",
            "bbox_height_mm",
            "bbox_width_mm",
            "major_axis_mm",
            "minor_axis_mm",
        ],
        "clinical_anatomical_names_confirmed": False,
        "level_pairing_source_runtime": "67C",
        "level_pairing_status":
            "EXPERIMENTAL_REVIEW_REQUIRED_NOT_FORMALLY_VALIDATED",
        "human_review_required": True,
    },
    "prohibited_interpretation": [
        "No diagnosis is produced.",
        "No stenosis grade is inferred from a morphometric threshold.",
        "No pathology is inferred from area or diameter alone.",
        "Axial raw classes must not be renamed clinically without documentary corroboration.",
    ],
    "scope": {
        "development_data": "OUTER_TRAIN_ONLY",
        "internal_test_accessed": False,
        "official_test_accessed": False,
        "training_performed": False,
    },
}

CONTRACT_PATH = METRICS_DIR / "68_E_measurement_contract.json"
CONTRACT_PATH.write_text(json.dumps(contract, indent=2))

api_example = {
    "studyId": "<runtime-study-id>",
    "plane": "sagittal|axial",
    "level": "L1-L2|L2-L3|L3-L4|L4-L5|L5-S1|null",
    "levelStatus":
        "validated|abstained|experimental_review_required",
    "structure":
        "disc_group_component|canal|vertebra_group_component|raw_*",
    "measurements": {
        "areaMm2": None,
        "bboxHeightMm": None,
        "bboxWidthMm": None,
        "majorAxisMm": None,
        "minorAxisMm": None,
    },
    "reviewRequired": True,
    "diagnostic": False,
}

API_PATH = METRICS_DIR / "68_E_api_example.json"
API_PATH.write_text(json.dumps(api_example, indent=2))

ready_contract = (
    sag_qc["status"] == "PASS"
    and ax_qc["status"] in ("PASS", "NO_ROWS")
)

summary = {
    "stage": "68-E",
    "run_studies": len(RUN_STUDY_IDS),
    "sagittal_rows": int(len(sag_df)),
    "axial_rows": int(len(ax_df)),
    "sagittal_failures": int(len(sag_fail_df)),
    "axial_failures": int(len(ax_fail_df)),
    "sagittal_qc": sag_qc["status"],
    "axial_qc": ax_qc["status"],
    "contract_sha256": sha256_file(CONTRACT_PATH),
    "api_sha256": sha256_file(API_PATH),
    "internal_test_accessed": False,
    "official_test_accessed": False,
    "training_performed": False,
    "ready_for_backend_contract_prototype": bool(ready_contract),
    "ready_for_production_level_integration": False,
    "production_blocker":
        "67C axial level pairing is not formally validated; axial clinical class names remain unconfirmed.",
}

SUM_PATH = METRICS_DIR / "68_E_final_summary.json"
SUM_PATH.write_text(json.dumps(summary, indent=2))

print("="*72)
print("68-E MORPHOMETRIC MEASUREMENT CONTRACT COMPLETE")
print("="*72)
print("Studies:", len(RUN_STUDY_IDS))
print("Sag rows:", len(sag_df), "failures:", len(sag_fail_df))
print("Ax rows:", len(ax_df), "failures:", len(ax_fail_df))
print("Sag QC:", sag_qc["status"])
print("Ax QC:", ax_qc["status"])
print("Contract SHA256:", sha256_file(CONTRACT_PATH))
print("API SHA256:", sha256_file(API_PATH))
print("Final summary SHA256:", sha256_file(SUM_PATH))
print(
    "READY FOR BACKEND CONTRACT PROTOTYPE:",
    summary["ready_for_backend_contract_prototype"],
)
print(
    "READY FOR PRODUCTION LEVEL INTEGRATION:",
    summary["ready_for_production_level_integration"],
)


68-E MORPHOMETRIC MEASUREMENT CONTRACT COMPLETE
Studies: 1382
Sag rows: 30085 failures: 1
Ax rows: 15107 failures: 0
Sag QC: PASS
Ax QC: PASS
Contract SHA256: 515e5bcc86a87b35897abfec205a4acb64adf425270ad525eb1d825ac77010bd
API SHA256: 07fa34833623ca75e2173df73a280a34f94a2fb46e0fd9c1e4b8766250211a01
Final summary SHA256: fde36fa6bc8272551bb0582b7e18ee72773622e1e157e3d80e8b8a4c0eb7a2d2
READY FOR BACKEND CONTRACT PROTOTYPE: True
READY FOR PRODUCTION LEVEL INTEGRATION: False
